## Import / setup


In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from datetime import datetime
from pathlib import Path

import pandas as pd
import numpy as np

# 將共整合目錄新增到路徑以匯入 get_data 模組
sys.path.insert(0, str(Path.cwd()))

from get_data import (
    create_bybit_session,
    get_all_bybit_usdt_spot_symbols,
    get_all_bybit_perp_symbols,
    fetch_sector_map_with_report,
    download_binance_sector_data,
    explore_downloaded_data,
    preview_market_data,
)

from coint import (
    load_price_matrix,
    scan_cointegrated_pairs,
    engle_granger_test,
    backtest_pair,
    summarize_backtest,
)

DATABASE_PATH = Path("data")
CONFIG_PATH = Path("config/api_config.json")  # 選填：用於 API 認證

START_DATE = datetime(2020, 1, 1)
END_DATE = datetime.now()

print(f"數據收集範圍: {START_DATE.date()} 至 {END_DATE.date()}")
print(f"資料庫路徑: {DATABASE_PATH}")
print("數據格式: Parquet")

try:
    bybit_session = create_bybit_session()
    print("Bybit 會話創建成功")
except ImportError as e:
    print(e)
    bybit_session = None

try:
    spot_symbols = get_all_bybit_usdt_spot_symbols()
    perp_symbols = get_all_bybit_perp_symbols()
    print(f"已獲取 {len(spot_symbols)} 個 Bybit 現貨 USDT 符號，範例: {spot_symbols[:5]}")
    print(f"已獲取 {len(perp_symbols)} 個 Bybit 永續 USDT 符號，範例: {perp_symbols[:5]}")
except Exception as e:
    print(f"獲取 Bybit 符號出錯: {e}")
    spot_symbols = []
    perp_symbols = []


# Data

## Fetch Sector Classification


In [ ]:
try:
    fetch_sector_map_with_report(
        base_path=DATABASE_PATH,
        categories_to_process=("spot", "linear"),
        sleep_seconds=0.5,
    )
except ImportError as e:
    print(e)
except Exception as e:
    print(f"獲取行業映射出錯: {e}")

## Download Binance Sector Data


In [ ]:
skip_sectors = ["USD Stablecoin", "Fiat-backed Stablecoin"]

try:
    binance_universe = download_binance_sector_data(
        base_path=DATABASE_PATH,
        start_date=START_DATE,
        end_date=END_DATE,
        skip_sectors=skip_sectors,
        interval="1h",
        sleep_seconds=0.5,
        include_funding_rate=False,
    )
except Exception as e:
    print(f"下載過程中出錯: {e}")

## Data Exploration and Validation


In [ ]:
data_summaries = explore_downloaded_data(DATABASE_PATH)

## Preview Parquet Data


In [ ]:
preview_market_data(DATABASE_PATH)

# Analysis

In [ ]:
# 讀取 Binance 現貨或合約價格矩陣
price_dir = DATABASE_PATH / "spot"
prices = load_price_matrix(price_dir, price_column="close", min_obs=500)
print(f"價格矩陣: {prices.shape[0]} 筆時間資料 x {prices.shape[1]} 個幣種")

# 掃描共整合候選配對
pair_candidates = scan_cointegrated_pairs(
    prices,
    max_pvalue=0.05,
    min_obs=500,
    top_n=20,
)
pair_candidates

In [ ]:
if not pair_candidates.empty:
    best_pair = pair_candidates.iloc[0]
    y_symbol = best_pair["y_symbol"]
    x_symbol = best_pair["x_symbol"]

    pair_backtest = backtest_pair(
        prices[y_symbol],
        prices[x_symbol],
        hedge_ratio=best_pair["hedge_ratio"],
        intercept=best_pair["intercept"],
        z_window=168,
        entry_z=2.0,
        exit_z=0.5,
        fee_rate=0.0004,
    )

    print(f"回測配對: {y_symbol} / {x_symbol}")
    print(summarize_backtest(pair_backtest))
    pair_backtest[["spread", "zscore", "position", "equity_curve"]].tail()
else:
    print("沒有找到符合條件的共整合配對。")